# Customer Churn — Responsible ML Baseline




In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

df = pd.read_csv('customer-churn-training-data.csv')
df


## 1. Data quality checks


In [ ]:
print('Shape:', df.shape)
print('\nMissing values:\n', df.isna().sum())
print('\nDuplicate customer IDs:', df['customer_id'].duplicated().sum())
print('\nTarget counts:\n', df['churned'].value_counts().sort_index())
print('\nCategorical values:', df['plan_type'].unique())


## 2. Simple descriptive analysis




In [ ]:
display(df.groupby('churned')[['tenure_months','support_tickets','monthly_spend','last_login_days']].mean())
display(pd.crosstab(df['plan_type'], df['churned']))


## 3. Majority-class baseline




In [ ]:
y = df['churned']
majority_pred = np.zeros(len(df), dtype=int)
print('Majority baseline accuracy:', accuracy_score(y, majority_pred))


## 4. Simple business-rule baseline




In [ ]:
rule_pred = (df['last_login_days'] >= 10).astype(int)
print('Rule accuracy:', accuracy_score(y, rule_pred))
print('Precision:', precision_score(y, rule_pred, zero_division=0))
print('Recall:', recall_score(y, rule_pred, zero_division=0))
print('F1:', f1_score(y, rule_pred, zero_division=0))
print('Confusion matrix:\n', confusion_matrix(y, rule_pred))


## 5. Logistic-regression baseline with leave-one-out evaluation


In [ ]:
X = df.drop(columns=['customer_id', 'churned'])
categorical = ['plan_type']
numeric = ['tenure_months', 'support_tickets', 'monthly_spend', 'last_login_days']

preprocess = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
])

pipe = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=2000))
])

loo = LeaveOneOut()
pred = cross_val_predict(pipe, X, y, cv=loo, method='predict')

print('LOOCV accuracy:', accuracy_score(y, pred))
print('LOOCV precision:', precision_score(y, pred, zero_division=0))
print('LOOCV recall:', recall_score(y, pred, zero_division=0))
print('LOOCV F1:', f1_score(y, pred, zero_division=0))
print('Confusion matrix:\n', confusion_matrix(y, pred))
print('\nClassification report:\n', classification_report(y, pred, zero_division=0))


## 6. Observations and limitations
- The dataset is very small (only 12 customers), so the evaluation results can change a lot with just one or two records.- The simple `last_login_days >= 10` rule performs well on this sample, so it is a useful benchmark for the ML model.- `customer_id` was excluded because it is an identifier rather than a meaningful predictor.- Accuracy alone is not enough to judge a churn model. I also looked at precision, recall, F1, and the confusion matrix.- Before using a model in a real business setting, I would need more data and a clearly defined prediction period.- I would also check that all features are available before the prediction is made so that information from after churn does not leak into the model.
